In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geemap
import ee

In [1]:

lake_features = [
        'lake_bottom_temperature',
        'lake_mix_layer_depth', 
        'lake_mix_layer_temperature',
        'lake_total_layer_temperature',
        'lake_bottom_temperature_min', 'lake_bottom_temperature_max',
        'lake_mix_layer_depth_min', 'lake_mix_layer_depth_max',
        'lake_mix_layer_temperature_min', 'lake_mix_layer_temperature_max',
        'lake_total_layer_temperature_min', 'lake_total_layer_temperature_max']
        
precipitation_features = [
        'total_precipitation_sum',
        'total_precipitation_min', 'total_precipitation_max',
        'runoff_sum',
        'runoff_min', 'runoff_max',
        'surface_runoff_sum',
        'surface_runoff_min', 'surface_runoff_max',
        'sub_surface_runoff_sum',
        'sub_surface_runoff_min', 'sub_surface_runoff_max']
        
evaporation_features = [
        'total_evaporation_sum',
        'total_evaporation_min', 'total_evaporation_max',
        'evaporation_from_open_water_surfaces_excluding_oceans_sum',
        'evaporation_from_open_water_surfaces_excluding_oceans_min',
        'evaporation_from_open_water_surfaces_excluding_oceans_max',
        'evaporation_from_bare_soil_sum']
        
thermal_features = [
        'skin_temperature',
        'temperature_2m',
        'dewpoint_temperature_2m',
        'surface_net_solar_radiation_sum',
        'surface_net_thermal_radiation_sum',
        'surface_sensible_heat_flux_sum',
        'surface_latent_heat_flux_sum',
        'surface_solar_radiation_downwards_sum',
        'surface_thermal_radiation_downwards_sum']
        
atmospheric_features = [
        'surface_pressure',
        'u_component_of_wind_10m',
        'v_component_of_wind_10m']
        
soil_features = [
        'volumetric_soil_water_layer_1',
        'volumetric_soil_water_layer_2',
        'volumetric_soil_water_layer_3',
        'volumetric_soil_water_layer_4']
        
bands = ['volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'lake_bottom_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_total_layer_temperature', 'lake_bottom_temperature_min', 'lake_bottom_temperature_max', 'lake_mix_layer_depth_min', 'lake_mix_layer_depth_max', 'lake_mix_layer_temperature_min', 'lake_mix_layer_temperature_max', 'lake_total_layer_temperature_min', 'lake_total_layer_temperature_max', 'total_precipitation_sum', 'total_precipitation_min', 'total_precipitation_max', 'runoff_sum', 'runoff_min', 'runoff_max', 'surface_runoff_sum', 'surface_runoff_min', 'surface_runoff_max', 'sub_surface_runoff_sum', 'sub_surface_runoff_min', 'sub_surface_runoff_max', 'total_evaporation_sum', 'total_evaporation_min', 'total_evaporation_max', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_min', 'evaporation_from_open_water_surfaces_excluding_oceans_max', 'evaporation_from_bare_soil_sum', 'skin_temperature', 'temperature_2m', 'dewpoint_temperature_2m', 'surface_net_solar_radiation_sum', 'surface_net_thermal_radiation_sum', 'surface_sensible_heat_flux_sum', 'surface_latent_heat_flux_sum', 'surface_solar_radiation_downwards_sum', 'surface_thermal_radiation_downwards_sum', 'surface_pressure', 'u_component_of_wind_10m', 'v_component_of_wind_10m']

In [3]:
bands = pd.read_csv("/home/desy/rift-waters/dataset/bogoria/raw/era5/era5_bogoria_2018-01-01_2024-12-31_merged.csv")

# Convert existing lists to sets for faster lookup
lake_set = set(lake_features)
precipitation_set = set(precipitation_features)
evaporation_set = set(evaporation_features)
thermal_set = set(thermal_features)
atmospheric_set = set(atmospheric_features)
soil_set = set(soil_features)

# Separate bands into respective lists
lake_bands = [band for band in bands if band in lake_set]
precipitation_bands = [band for band in bands if band in precipitation_set]
evaporation_bands = [band for band in bands if band in evaporation_set]
thermal_bands = [band for band in bands if band in thermal_set]
atmospheric_bands = [band for band in bands if band in atmospheric_set]
soil_bands = [band for band in bands if band in soil_set]

print("Lake bands:", lake_bands)
print("Precipitation bands:", precipitation_bands)
print("Evaporation bands:", evaporation_bands)
print("Thermal bands:", thermal_bands)
print("Atmospheric bands:", atmospheric_bands)
print("Soil bands:", soil_bands)

Lake bands: ['lake_bottom_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_total_layer_temperature', 'lake_bottom_temperature_min', 'lake_bottom_temperature_max', 'lake_mix_layer_depth_min', 'lake_mix_layer_depth_max', 'lake_mix_layer_temperature_min', 'lake_mix_layer_temperature_max', 'lake_total_layer_temperature_min', 'lake_total_layer_temperature_max']
Precipitation bands: ['total_precipitation_sum', 'total_precipitation_min', 'total_precipitation_max', 'runoff_sum', 'runoff_min', 'runoff_max', 'surface_runoff_sum', 'surface_runoff_min', 'surface_runoff_max', 'sub_surface_runoff_sum', 'sub_surface_runoff_min', 'sub_surface_runoff_max']
Evaporation bands: ['total_evaporation_sum', 'total_evaporation_min', 'total_evaporation_max', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_min', 'evaporation_from_open_water_surfaces_excluding_oceans_max', 'evaporation_from_bare_soil_sum']
Thermal bands: 

In [ ]:
import pandas as pd

era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .select(bands)
        .filterBounds(ROI)
        .filterDate(start_date, end_date))

def add_mean_precip(image):
    mean = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ROI,
        scale=9000,
        bestEffort=True
    )
    # image.set() accepts a dictionary directly — sets all bands at once
    return image.set(mean).set('system:time_start', image.get('system:time_start'))

daily_averages = era5.map(add_mean_precip)

# Pull dates once
dates = daily_averages.aggregate_array('system:time_start').getInfo()
dates = pd.to_datetime(dates, unit='ms')

# Pull each band separately — aggregate_array can't take a list
data = {}
for band in bands:
    data[band] = daily_averages.aggregate_array(band).getInfo()

daily_averages_df = pd.DataFrame(data, index=dates)
daily_averages_df.index.name = 'date'
daily_averages_df.head()

In [ ]:
import pandas as pd

# Load the collection
era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .select('total_precipitation_sum',  ).filterBounds(ROI)
        .filterDate(start_date, end_date))

# Define your polygon


# Define the function to add mean precipitation as a property
def add_mean_precip(image):
    mean = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ROI,
        scale=9000,
        bestEffort=True
    )
    return image.set('mean_precip', mean.get('total_precipitation_sum'))

# Apply the function to the collection
daily_averages = era5.map(add_mean_precip)
print(daily_averages.getInfo())
daily_averages_list = daily_averages.aggregate_array('mean_precip').getInfo()
daily_averages_df = pd.DataFrame(daily_averages_list, columns=['mean_precipitation'])
#daily_averages_df.to_csv("era5_daily_precipitation.csv")
daily_averages_df